# Underwater RL Image Enhancement

Notebook finale, breve ma completo, per spiegare il progetto in modo lineare e presentabile.

Cosa mostra:
- da quale idea partiamo rispetto a Bologna;
- come viene trattata un'immagine underwater degradata;
- quali operazioni puo' applicare l'agente;
- come leggere i risultati della **run ufficiale finale v4.0**.


## 1. Idea del progetto

Questo progetto migliora immagini underwater degradate con **Double DQN (DDQN)**.

Le idee chiave sono:
- il progetto e' **ispirato** al lavoro di Bologna 2022;
- il codice qui e' stato **riscritto da zero**;
- non stiamo usando un modello fisico completo dell'acqua;
- usiamo pero' trasformazioni classiche e interpretabili, applicate in sequenza.

Nel workflow ufficiale finale `v4.0` l'agente lavora con:
- 4 azioni: `white_balance`, `contrast_up`, `sharpen`, `stop`;
- massimo 5 step per immagine;
- dataset UIEB paired: immagine degradata + reference reale.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "README.md").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.actions import ACTION_SETS, apply_action_to_pil, get_action_descriptions
from src.metrics import compute_psnr, compute_ssim

print("Repository root:", REPO_ROOT)

## 2. Configurazione minima

Il notebook prova a usare automaticamente:
- la **run full** più recente con report completi;
- una coppia reale di immagini UIEB.

Se vuoi forzare path specifici, imposta i valori qui sotto.

In [ ]:
PREFERRED_RUN_ID = "dqn_underwater_full_20260510_165955_1494"
RUN_DIR_OVERRIDE = None
UIEB_ROOT_OVERRIDE = None
SAMPLE_DEGRADED_PATH = None
SAMPLE_REFERENCE_PATH = None
SAMPLE_PAIR_INDEX = 0
ACTION_SET_NAME = "underwater_curated_v1"


In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def find_official_or_latest_full_run(repo_root: Path, preferred_run_id: str | None = None) -> Path | None:
    roots = [repo_root / "logs", Path(os.getenv("LOGS_ROOT", "logs"))]
    candidates = []
    preferred_match = None
    for base in roots:
        dqn_dir = base / "dqn"
        if not dqn_dir.exists():
            continue
        for run_dir in dqn_dir.glob("dqn_underwater_full_*"):
            if not run_dir.is_dir():
                continue
            required = [
                run_dir / "underwater_results_summary.json",
                run_dir / "evaluation_baselines_best.json",
                run_dir / "evaluation_baselines_final.json",
                run_dir / "action_analysis_best.json",
                run_dir / "action_analysis_final.json",
            ]
            if all(path.exists() for path in required):
                if preferred_run_id and run_dir.name == preferred_run_id:
                    preferred_match = run_dir
                candidates.append((run_dir.stat().st_mtime, run_dir))
    if preferred_match is not None:
        return preferred_match
    if not candidates:
        return None
    candidates.sort()
    return candidates[-1][1]


def find_uieb_root() -> Path | None:
    candidates = [UIEB_ROOT_OVERRIDE, os.getenv("UIEB_ROOT")]
    dataset_root = os.getenv("DATASET_ROOT")
    if dataset_root:
        candidates.append(str(Path(dataset_root) / "UIEB"))
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser()
        if (path / "raw").exists() and (path / "reference").exists():
            return path
    return None


def collect_pairs(uieb_root: Path):
    rows = []
    for degraded_path in sorted((uieb_root / "raw").iterdir()):
        if degraded_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
            continue
        reference_path = uieb_root / "reference" / degraded_path.name
        if reference_path.exists():
            rows.append({
                "image_id": degraded_path.stem,
                "degraded": degraded_path,
                "reference": reference_path,
            })
    return rows


def open_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def show_images(images, titles=None, figsize=(16, 5)):
    titles = titles or [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def image_stats(image: Image.Image) -> dict[str, float]:
    arr = np.asarray(image).astype(np.float32)
    return {
        "mean_brightness": float(arr.mean()),
        "mean_red": float(arr[..., 0].mean()),
        "mean_green": float(arr[..., 1].mean()),
        "mean_blue": float(arr[..., 2].mean()),
        "std_intensity": float(arr.std()),
    }


def absolute_difference_image(img_a: Image.Image, img_b: Image.Image) -> Image.Image:
    a = np.asarray(img_a.resize(img_b.size)).astype(np.float32)
    b = np.asarray(img_b).astype(np.float32)
    diff = np.abs(a - b).clip(0, 255).astype(np.uint8)
    return Image.fromarray(diff)


RUN_DIR = Path(RUN_DIR_OVERRIDE).expanduser() if RUN_DIR_OVERRIDE else find_official_or_latest_full_run(REPO_ROOT, PREFERRED_RUN_ID)
UIEB_ROOT = find_uieb_root()

print("Run directory:", RUN_DIR)
print("UIEB root:", UIEB_ROOT)


## 3. Come lavora su un'immagine degradata

La pipeline, immagine per immagine, è questa:

1. entra una foto underwater degradata;
2. l'ambiente la confronta con la reference solo per misurare il miglioramento;
3. l'agente sceglie una piccola azione;
4. l'immagine viene aggiornata;
5. si misura se PSNR e SSIM sono migliorati;
6. il processo continua per pochi step o si ferma con `stop`.

Qui sotto vediamo una coppia reale `degraded/reference`.

In [ ]:
sample_record = None

if SAMPLE_DEGRADED_PATH and SAMPLE_REFERENCE_PATH:
    sample_record = {
        "image_id": Path(SAMPLE_DEGRADED_PATH).stem,
        "degraded": Path(SAMPLE_DEGRADED_PATH).expanduser(),
        "reference": Path(SAMPLE_REFERENCE_PATH).expanduser(),
    }
elif UIEB_ROOT is not None:
    pairs = collect_pairs(UIEB_ROOT)
    if pairs:
        sample_record = pairs[SAMPLE_PAIR_INDEX % len(pairs)]

if sample_record is None:
    print("Immagini non trovate. Imposta UIEB_ROOT oppure SAMPLE_DEGRADED_PATH/SAMPLE_REFERENCE_PATH.")
else:
    degraded_image = open_rgb(sample_record["degraded"])
    reference_image = open_rgb(sample_record["reference"])
    diff_image = absolute_difference_image(degraded_image, reference_image)

    show_images(
        [degraded_image, reference_image, diff_image],
        ["Input degradato", "Reference", "Differenza assoluta"],
        figsize=(18, 5),
    )

    stats_df = pd.DataFrame([
        {"image": "degraded", **image_stats(degraded_image)},
        {"image": "reference", **image_stats(reference_image)},
    ])
    display(stats_df)
    print("PSNR input vs reference:", round(compute_psnr(degraded_image, reference_image), 4))
    print("SSIM input vs reference:", round(compute_ssim(degraded_image, reference_image), 4))

## 4. Quali operazioni può fare l'agente

Nel setup ufficiale usiamo un set piccolo e interpretabile. Questo aiuta a capire meglio il comportamento del modello.

In [ ]:
action_rows = []
for action_id, action_name in ACTION_SETS[ACTION_SET_NAME]["action_names"].items():
    action_rows.append({
        "action_id": action_id,
        "action_name": action_name,
        "description": get_action_descriptions(ACTION_SET_NAME).get(action_id, ""),
    })
display(pd.DataFrame(action_rows))

In [ ]:
if sample_record is None:
    print("Serve prima una coppia di immagini reale.")
else:
    preview_rows = []
    preview_images = [degraded_image]
    preview_titles = ["input"]

    base_psnr = compute_psnr(degraded_image, reference_image)
    base_ssim = compute_ssim(degraded_image, reference_image)

    for action_id, action_name in ACTION_SETS[ACTION_SET_NAME]["action_names"].items():
        if action_name == "stop":
            continue
        transformed = apply_action_to_pil(degraded_image, action_id, ACTION_SET_NAME)
        psnr_now = compute_psnr(transformed, reference_image)
        ssim_now = compute_ssim(transformed, reference_image)
        preview_images.append(transformed)
        preview_titles.append(f"{action_name}\nΔPSNR {psnr_now - base_psnr:+.3f}")
        preview_rows.append({
            "action": action_name,
            "psnr_after": psnr_now,
            "delta_psnr_vs_input": psnr_now - base_psnr,
            "ssim_after": ssim_now,
            "delta_ssim_vs_input": ssim_now - base_ssim,
        })

    show_images(preview_images, preview_titles, figsize=(18, 5))
    display(pd.DataFrame(preview_rows).sort_values("delta_psnr_vs_input", ascending=False))

## 5. Esempio di mini-rollout

Qui simuliamo una breve sequenza di azioni su un'immagine reale, per far vedere come l'immagine venga aggiornata passo dopo passo.

In [ ]:
MANUAL_SEQUENCE = ["white_balance", "contrast_up", "sharpen"]

if sample_record is None:
    print("Serve prima una coppia di immagini reale.")
else:
    name_to_id = {v: k for k, v in ACTION_SETS[ACTION_SET_NAME]["action_names"].items()}
    current = degraded_image.copy()
    rollout_images = [current]
    rollout_titles = ["step 0\ninput"]
    rollout_rows = []

    prev_psnr = compute_psnr(current, reference_image)
    prev_ssim = compute_ssim(current, reference_image)

    for step_idx, action_name in enumerate(MANUAL_SEQUENCE, start=1):
        action_id = name_to_id[action_name]
        current = apply_action_to_pil(current, action_id, ACTION_SET_NAME)
        psnr_now = compute_psnr(current, reference_image)
        ssim_now = compute_ssim(current, reference_image)
        rollout_images.append(current)
        rollout_titles.append(f"step {step_idx}\n{action_name}\nΔPSNR {psnr_now - prev_psnr:+.3f}")
        rollout_rows.append({
            "step": step_idx,
            "action": action_name,
            "psnr": psnr_now,
            "delta_psnr_vs_prev": psnr_now - prev_psnr,
            "ssim": ssim_now,
            "delta_ssim_vs_prev": ssim_now - prev_ssim,
        })
        prev_psnr = psnr_now
        prev_ssim = ssim_now

    show_images(rollout_images, rollout_titles, figsize=(18, 5))
    display(pd.DataFrame(rollout_rows))

## 6. Risultati della run ufficiale finale v4.0

Qui leggiamo gli artifact della **run ufficiale finale** e li usiamo per rispondere a tre domande semplici:

1. Migliora davvero le immagini paired?
2. Supera Bologna come risultato assoluto?
3. Quale configurazione finale abbiamo scelto e perche'?


In [ ]:
if RUN_DIR is None:
    print("Run full non trovata.")
else:
    run_meta = load_json(RUN_DIR / "run_meta.json")
    best_eval = load_json(RUN_DIR / "evaluation_baselines_best.json")
    final_eval = load_json(RUN_DIR / "evaluation_baselines_final.json")
    summary = load_json(RUN_DIR / "underwater_results_summary.json")
    effective_config = load_json(RUN_DIR / "effective_config.json") if (RUN_DIR / "effective_config.json").exists() else {}
    env_cfg = effective_config.get("environment", {})
    reward_cfg = effective_config.get("reward", {})

    config_df = pd.DataFrame([
        {"setting": "algorithm", "value": "DDQN"},
        {"setting": "dataset", "value": "UIEB paired"},
        {"setting": "action_set", "value": summary["action_set_name"]},
        {"setting": "num_actions", "value": summary["num_actions"]},
        {"setting": "max_steps", "value": env_cfg.get("max_steps", "n/a")},
        {"setting": "include_lab_stats", "value": env_cfg.get("include_lab_stats", False)},
        {"setting": "psnr_weight", "value": reward_cfg.get("psnr_weight", "n/a")},
        {"setting": "ssim_weight", "value": reward_cfg.get("ssim_weight", "n/a")},
        {"setting": "num_episodes", "value": summary["num_episodes"]},
    ])
    display(config_df)

    overview_df = pd.DataFrame([
        {"checkpoint": "best", "episode": summary["best_checkpoint"]["episode"], "mean_delta_psnr": summary["best_checkpoint"]["id_mean_delta_psnr"], "output_psnr": summary["best_checkpoint"]["output_psnr"], "output_ssim": summary["best_checkpoint"]["output_ssim"], "stop_rate": best_eval["stop_rate"], "dominant_action_share": best_eval["dominant_action_share"], "acceptance": summary["best_checkpoint"]["acceptance_passed"]},
        {"checkpoint": "final", "episode": "-", "mean_delta_psnr": summary["final_checkpoint"]["id_mean_delta_psnr"], "output_psnr": summary["final_checkpoint"]["output_psnr"], "output_ssim": summary["final_checkpoint"]["output_ssim"], "stop_rate": final_eval["stop_rate"], "dominant_action_share": final_eval["dominant_action_share"], "acceptance": summary["final_checkpoint"]["acceptance_passed"]},
    ])
    display(overview_df)

    print("Run id:", summary["run_id"])
    print("Best checkpoint episode:", summary["best_checkpoint"]["episode"])
    print("Best checkpoint selection metric:", run_meta["best_by_metric"])


In [ ]:
if RUN_DIR is None:
    print("Run full non trovata.")
else:
    summary = load_json(RUN_DIR / "underwater_results_summary.json")
    absolute_df = pd.DataFrame([
        {"system": "Bologna DDQN (2022)", "output_psnr": summary["bologna_reference"]["psnr"], "output_ssim": summary["bologna_reference"]["ssim"], "episodes": summary["bologna_reference"]["episodes"], "num_actions": summary["bologna_reference"]["actions"]},
        {"system": "Ours best checkpoint", "output_psnr": summary["best_checkpoint"]["output_psnr"], "output_ssim": summary["best_checkpoint"]["output_ssim"], "episodes": summary["num_episodes"], "num_actions": summary["num_actions"]},
        {"system": "Ours final checkpoint", "output_psnr": summary["final_checkpoint"]["output_psnr"], "output_ssim": summary["final_checkpoint"]["output_ssim"], "episodes": summary["num_episodes"], "num_actions": summary["num_actions"]},
    ])
    display(absolute_df)

    quality_df = pd.DataFrame([
        {"view": "best ID", "mean_delta_psnr": summary["best_checkpoint"]["id_mean_delta_psnr"], "mean_delta_ssim": summary["best_checkpoint"]["id_mean_delta_ssim"]},
        {"view": "final ID", "mean_delta_psnr": summary["final_checkpoint"]["id_mean_delta_psnr"], "mean_delta_ssim": summary["final_checkpoint"]["id_mean_delta_ssim"]},
        {"view": "OOD challenging-60", "mean_delta_psnr": summary["ood_challenging60"]["mean_delta_uciqe"], "mean_delta_ssim": summary["ood_challenging60"]["mean_delta_uiqm_proxy"]},
    ])
    display(quality_df)

    ablation_run_ids = {
        "Baseline v3.0": "dqn_underwater_full_20260510_104821_1490",
        "Ablation A: max_steps=5": "dqn_underwater_full_20260510_111515_1491",
        "Ablation B: extended actions": "dqn_underwater_full_20260510_155811_1492",
        "Ablation C: LAB stats": "dqn_underwater_full_20260510_162433_1493",
        "Official v4.0": summary["run_id"],
    }
    ablation_rows = []
    for label, run_id in ablation_run_ids.items():
        run_path = RUN_DIR.parent / run_id / "underwater_results_summary.json"
        if not run_path.exists():
            continue
        run_summary = load_json(run_path)
        ablation_rows.append({
            "run": label,
            "best_delta_psnr": run_summary["best_checkpoint"]["id_mean_delta_psnr"],
            "best_output_psnr": run_summary["best_checkpoint"]["output_psnr"],
            "ood_delta_uciqe": run_summary["ood_challenging60"]["mean_delta_uciqe"],
            "ood_delta_uiqm": run_summary["ood_challenging60"]["mean_delta_uiqm_proxy"],
        })
    display(pd.DataFrame(ablation_rows))

    print("Lettura semplice dei risultati:")
    print("- Su test ID paired il modello migliora davvero le immagini: il delta PSNR del best checkpoint e' +{:.4f} dB.".format(summary["best_checkpoint"]["id_mean_delta_psnr"]))
    print("- Sul confronto assoluto, il best checkpoint resta sopra Bologna in PSNR e SSIM.")
    print("- La configurazione finale scelta e' max_steps=5 con action set curated: e' la piu' forte senza complicare troppo il sistema.")
    print("- L'OOD migliora rispetto alla baseline v3.0, ma resta la parte del progetto che ha ancora piu' margine di crescita.")


## 8. Messaggio finale

Il progetto finale e' coerente con questo racconto:

- partiamo da Bologna come ispirazione;
- abbiamo riscritto pipeline e codice in modo piu' controllato;
- usiamo DDQN su immagini reali UIEB paired;
- l'immagine degradata viene corretta con piccole azioni interpretabili;
- la configurazione finale `v4.0` usa 4 azioni e 5 step massimi;
- il miglioramento ID e' forte e leggibile;
- anche l'OOD e' migliorato rispetto alla baseline iniziale, ma resta il margine principale per lavoro futuro.
